# CIFAR-100 quick run (Colab)

MAEG3080 project — **SE-ResNet** path: baseline (if needed) → short finetune → **official test** via `scripts/quick_se_resnet_results.sh`.

**Setup:** Runtime → **GPU** (T4 or better). Run cells **top to bottom**.

**On your Mac:** `bash colab/prepare_colab_bundle.sh` → upload `artifacts/cifar_hydra_project.tar.gz` to Google Drive under **`My Drive/Colab_CIFAR/`** (or My Drive root), then run the unpack cell.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import os
import pathlib
import shutil
import sys
import tarfile

WORK_ROOT = pathlib.Path('/content')
PROJECT_ROOT = WORK_ROOT / 'cifar-hydra'
MY_DRIVE = pathlib.Path('/content/drive/MyDrive')
DRIVE_ROOT = MY_DRIVE / 'Colab_CIFAR'
BUNDLE_NAME = 'cifar_hydra_project.tar.gz'

MANUAL_BUNDLE = None
# MANUAL_BUNDLE = pathlib.Path('/content/drive/MyDrive/Colab_CIFAR/cifar_hydra_project.tar.gz')


def resolve_bundle_path():
    if MANUAL_BUNDLE is not None and MANUAL_BUNDLE.is_file():
        return MANUAL_BUNDLE
    candidates = [DRIVE_ROOT / BUNDLE_NAME, MY_DRIVE / BUNDLE_NAME]
    for p in candidates:
        if p.is_file():
            return p
    if MY_DRIVE.is_dir():
        for p in MY_DRIVE.rglob(BUNDLE_NAME):
            if p.is_file():
                return p
    return None


BUNDLE_PATH = resolve_bundle_path()
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

if BUNDLE_PATH is None or not BUNDLE_PATH.exists():
    raise FileNotFoundError(
        f'Missing {BUNDLE_NAME}. Upload it to My Drive/Colab_CIFAR/ (see intro).'
    )

with tarfile.open(BUNDLE_PATH, 'r:gz') as archive:
    if sys.version_info >= (3, 12):
        archive.extractall(PROJECT_ROOT, filter='data')
    else:
        archive.extractall(PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print('Bundle:', BUNDLE_PATH)
print('Project:', PROJECT_ROOT)
print('Drive root:', DRIVE_ROOT)


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


In [ ]:
import subprocess
import torch

subprocess.run(['nvidia-smi'], check=False)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import os
import pathlib
import sys
import torch

DATA_DIR = DRIVE_ROOT / 'data'
SAVE_DIR = DRIVE_ROOT / 'checkpoints'
OUTPUT_DIR = DRIVE_ROOT / 'outputs'

DATA_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ['PYTHON_BIN'] = sys.executable
os.environ['DATA_DIR'] = str(DATA_DIR)
os.environ['SAVE_DIR'] = str(SAVE_DIR)
os.environ['DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['NUM_WORKERS'] = '4'
os.environ['BATCH_SIZE'] = '128'

print('DATA_DIR =', DATA_DIR)
print('SAVE_DIR =', SAVE_DIR)
print('DEVICE =', os.environ['DEVICE'])


## Train + official test

Runs **`quick_se_resnet_results.sh`**: 4-epoch baseline if missing → **30-epoch** finetune → **`results.py`** on the test set.


In [ ]:
import os
import subprocess
import sys

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHON_BIN'] = sys.executable
env['DATA_DIR'] = str(DATA_DIR)
env['SAVE_DIR'] = str(SAVE_DIR)
env['DEVICE'] = os.environ['DEVICE']
env['NUM_WORKERS'] = '4'
env['TQDM_DISABLE'] = '1'
# env['EPOCHS'] = '60'  # stronger finetune (slower)
# env['BATCH_SIZE'] = '128'  # if OOM

process = subprocess.Popen(
    ['bash', 'scripts/quick_se_resnet_results.sh'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, ['bash', 'scripts/quick_se_resnet_results.sh'])
